# 🧬 Exp 4: Population Graph Convolutional Network (Population GCN) cho Phân Loại Tự Kỷ (ASD)
## Tái Lập và Phát Triển Dựa Trên Bài Báo Parisot et al. (Medical Image Analysis 2018 / arXiv:1806.01738)

---

### 📌 Bối cảnh và Động lực:
Trong thực nghiệm trước (**Exp 3 — Brain Graph GNN**), việc phân loại từng đồ thị não riêng lẻ với node features bị nén thành 2 con số (`mean`, `std`) và dùng `global_mean_pool` đã dẫn tới hiện tượng **mất hoàn toàn đặc trưng**, khiến mô hình chỉ đạt **~49.4% - 50.6% Balanced Accuracy** (tương đương đoán ngẫu nhiên).

Trong notebook này, chúng ta chuyển sang hướng tiếp cận chuẩn mực của **Parisot et al. (2018)**:
1. **Mỗi bệnh nhân là 1 nút (Node)** trong một Đồ thị Dân số duy nhất (**Population Graph**).
2. **Đặc trưng mỗi nút (Node Features)**: Toàn bộ vector liên kết chức năng não (**19,900 cạnh FC** từ Atlas CC200), được chọn lọc khoa học bằng **Feature Selection trong từng Fold** (tránh triệt để Data Leakage).
3. **Cạnh đồ thị (Edges)**: Tích hợp thông tin đa phương thức (**Phenotypic Metadata**: `SITE_ID`, `SEX`, `AGE_AT_SCAN`) nhân với độ tương đồng hình ảnh não bộ (**Cosine Similarity**), sau đó làm thưa bằng **k-NN**.
4. **Mô hình GNN**: Triển khai **Spectral GCN** (Kipf-Welling) tối ưu hóa trực tiếp trên ma trận kề chuẩn hóa, tự động tăng tốc trên GPU.
5. **Thẩm định**: Chạy song song cả **10-Fold CV** (so với mốc 70.4% của bài báo) và **5-Fold CV** (so với mốc 64.5% của SVM Baseline).


In [ ]:
# 1. Kiểm tra môi trường & Phần cứng
import torch
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Thiết bị: {torch.cuda.get_device_name(0)}")

# Tự động chọn Device GPU/CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Môi trường thực thi mặc định: {DEVICE}")


---
## 📦 2. Nạp Dữ Liệu ABIDE-I (CC200 Atlas) & Phenotypic Metadata

Chúng ta tải dữ liệu từ 871 subjects đã qua kiểm duyệt chất lượng (Quality Checked) từ ABIDE-I:
- **Imaging Data**: Ma trận Functional Connectivity $200 \times 200$, trích xuất tam giác trên thành vector 19,900 chiều.
- **Phenotypic Data**: `SITE_ID` (Trung tâm quét), `SEX` (Giới tính: 1=Nam, 2=Nữ), `AGE_AT_SCAN` (Tuổi khi chụp).


In [ ]:
import os
import glob

def load_abide_population_data():
    # Kiểm tra nếu chạy trên Kaggle hoặc Local
    kaggle_npz = '/kaggle/input/datasets/kietkolat/fc-matric-for-graph/abide_cc200_fc_matrices.npz'
    local_data_dir = 'data'
    
    # 1. Tải Phenotypic metadata
    pheno_candidates = [
        os.path.join(local_data_dir, 'ABIDE_pcp', 'Phenotypic_V1_0b_preprocessed1.csv'),
        '../data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv',
        '/kaggle/input/abide-pcp/Phenotypic_V1_0b_preprocessed1.csv'
    ]
    pheno_path = next((p for p in pheno_candidates if os.path.exists(p)), None)
    
    if pheno_path is None:
        from nilearn.datasets import fetch_abide_pcp
        print("Đang tải dữ liệu Phenotypic từ Nilearn...")
        abide = fetch_abide_pcp(data_dir=local_data_dir, quality_checked=True)
        df_pheno_raw = abide.phenotypic
    else:
        df_pheno_raw = pd.read_csv(pheno_path)
    
    # 2. Tải FC Features
    if os.path.exists(kaggle_npz):
        print(f"Nạp từ NPZ: {kaggle_npz}")
        data = np.load(kaggle_npz, allow_pickle=True)
        fc_matrices = data['fc_matrices']
        y = data['y']
        triu_idx = np.triu_indices(fc_matrices.shape[1], k=1)
        features = np.array([fc[triu_idx] for fc in fc_matrices])
        df_pheno = df_pheno_raw.iloc[:len(y)].copy()
        return features, df_pheno, y

    # Tải từ các file ROI 1D cục bộ
    raw_1d_dirs = [
        os.path.join(local_data_dir, 'ABIDE_pcp', 'cpac', 'filt_noglobal'),
        '../data/ABIDE_pcp/cpac/filt_noglobal'
    ]
    raw_dir = next((d for d in raw_1d_dirs if os.path.exists(d)), None)
    
    if raw_dir and len(glob.glob(os.path.join(raw_dir, '*cc200*.1D'))) > 0:
        files = sorted(glob.glob(os.path.join(raw_dir, '*cc200*.1D')))
        print(f"Tìm thấy {len(files)} file ROI 1D.")
        
        valid_sub_ids = []
        valid_files = []
        for f in files:
            parts = os.path.basename(f).split('_')
            sub_id = int(parts[1]) if len(parts) >= 2 else None
            if sub_id and sub_id in df_pheno_raw['SUB_ID'].values:
                valid_sub_ids.append(sub_id)
                valid_files.append(f)
                
        df_pheno = df_pheno_raw[df_pheno_raw['SUB_ID'].isin(valid_sub_ids)].copy()
        df_pheno.sort_values('SUB_ID', inplace=True)
        df_pheno.reset_index(drop=True, inplace=True)
        
        triu_idx = np.triu_indices(200, k=1)
        features_list, labels_list = [], []
        
        for _, row in df_pheno.iterrows():
            sub_id = row['SUB_ID']
            match_file = [f for f in valid_files if f"_{sub_id:07d}_" in f or f"_{sub_id}_" in f][0]
            ts = np.loadtxt(match_file)
            corr = np.corrcoef(ts.T)
            np.nan_to_num(corr, copy=False, nan=0.0)
            features_list.append(corr[triu_idx])
            labels_list.append(1 if row['DX_GROUP'] == 1 else 0)
            
        features = np.array(features_list)
        y = np.array(labels_list)
        return features, df_pheno, y

    raise FileNotFoundError("Không tìm thấy dữ liệu fMRI ABIDE!")

features, df_pheno, y = load_abide_population_data()
print("Dữ liệu hoàn tất:")
print(f"   - Số lượng bệnh nhân (Nodes): {features.shape[0]}")
print(f"   - Số chiều đặc trưng FC gốc: {features.shape[1]} (200 * 199 / 2)")
print(f"   - Phân bố nhãn: ASD={np.sum(y == 1)}, TD={np.sum(y == 0)}")


---
## 🕸️ 3. Xây Dựng Population Graph (Parisot et al., 2018)

Công thức xác định trọng số cạnh $W(i, j)$ giữa hai cá thể $i$ và $j$:

$$W(i, j) = \text{Sim}(x_i, x_j) \cdot \sum_{h=1}^H \gamma(M_h(i), M_h(j))$$

Trong đó:
- $\text{Sim}(x_i, x_j)$: Độ tương đồng Cosine giữa đặc trưng FC của hai bệnh nhân: $\frac{x_i \cdot x_j}{\|x_i\| \|x_j\|}$.
- $\gamma_{\text{site}}(i, j) = \delta(\text{Site}_i, \text{Site}_j)$: Trọng số bằng 1 nếu cùng trung tâm quét, 0 nếu khác.
- $\gamma_{\text{sex}}(i, j) = \delta(\text{Sex}_i, \text{Sex}_j)$: Trọng số bằng 1 nếu cùng giới tính, 0 nếu khác.
- $\gamma_{\text{age}}(i, j) = \mathbb{I}(|\text{Age}_i - \text{Age}_j| \le 2)$: Trọng số bằng 1 nếu chênh lệch tuổi $\le 2$ năm.
- **Sparsification**: Giữ lại $k$ láng giềng gần nhất ($k=10$) cho mỗi nút để chống over-smoothing.


In [ ]:
def build_population_graph(features, pheno_df, k_nn=10, age_threshold=2.0):
    n_nodes = len(features)
    print(f"Đang tính ma trận kề cho {n_nodes} đối tượng...")
    
    # 1. Cosine similarity của imaging features
    norm_feat = features / (np.linalg.norm(features, axis=1, keepdims=True) + 1e-8)
    sim_matrix = np.dot(norm_feat, norm_feat.T)
    sim_matrix = 0.5 * (sim_matrix + 1.0)  # Chuẩn hóa về [0, 1]
    
    # 2. Phenotypic similarity measures
    sites = pheno_df['SITE_ID'].values
    sexes = pheno_df['SEX'].values
    ages  = pheno_df['AGE_AT_SCAN'].values
    
    gamma_site = (sites[:, None] == sites[None, :]).astype(float)
    gamma_sex  = (sexes[:, None] == sexes[None, :]).astype(float)
    gamma_age  = (np.abs(ages[:, None] - ages[None, :]) <= age_threshold).astype(float)
    
    pheno_weight = gamma_site + gamma_sex + gamma_age
    
    # Ma trận tương đồng tổng hợp
    W = sim_matrix * pheno_weight
    np.fill_diagonal(W, 0.0)
    
    # 3. k-NN Sparsification
    adj_sparse = np.zeros_like(W)
    for i in range(n_nodes):
        top_k_idx = np.argsort(W[i])[-k_nn:]
        adj_sparse[i, top_k_idx] = W[i, top_k_idx]
        
    # Đối xứng hóa: A = max(A, A^T)
    adj_symmetric = np.maximum(adj_sparse, adj_sparse.T)
    np.fill_diagonal(adj_symmetric, 1.0)
    
    # Chuẩn hóa đối xứng D^{-1/2} A D^{-1/2} cho GCN (Kipf & Welling)
    row_sum = np.array(adj_symmetric.sum(1))
    d_inv_sqrt = np.power(row_sum, -0.5).flatten()
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)
    adj_norm = d_mat_inv_sqrt.dot(adj_symmetric).dot(d_mat_inv_sqrt)
    
    density = np.count_nonzero(adj_symmetric) / (n_nodes * n_nodes)
    print(f"Population Graph hoàn thành: {n_nodes} nút, mật độ cạnh: {density:.2%}")
    return adj_symmetric, adj_norm

adj_symmetric, adj_norm = build_population_graph(features, df_pheno, k_nn=10)

# Visualize một góc của ma trận kề
plt.figure(figsize=(7, 6))
plt.imshow(adj_symmetric[:100, :100], cmap='viridis', aspect='auto')
plt.colorbar(label='Trọng số cạnh W')
plt.title('Góc Ma Trận Kề Population Graph (100 đối tượng đầu tiên)')
plt.xlabel('Đối tượng j')
plt.ylabel('Đối tượng i')
plt.show()


---
## 🧠 4. Kiến Trúc Mô Hình: Spectral Population GCN

Chúng ta sử dụng lớp **Graph Convolution chuẩn mực của Kipf & Welling (ICLR 2017)**:

$$H^{(l+1)} = \sigma\left(\tilde{D}^{-\frac{1}{2}} \tilde{A} \tilde{D}^{-\frac{1}{2}} H^{(l)} W^{(l)}\right)$$

- Tối ưu hóa trực tiếp trên ma trận kề chuẩn hóa `adj_norm` kích thước $(871 \times 871)$.
- Chạy siêu tốc trên GPU với phép nhân ma trận `torch.matmul`, loại bỏ hoàn toàn các lỗi xung đột kích thước Tensor thường gặp trong PyG.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class NativeGCNConv(nn.Module):
    """
    Lớp GCN Kipf & Welling chuẩn: H' = adj_norm @ H @ W
    Hoạt động với ma trận kề đối xứng đã chuẩn hóa (N, N).
    """
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features)) if bias else None
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)
            
    def forward(self, x, adj_norm):
        # x: (N, in_features), adj_norm: (N, N)
        support = torch.matmul(x, self.weight)
        output = torch.matmul(adj_norm, support)
        if self.bias is not None:
            output = output + self.bias
        return output

class PopulationGCN(nn.Module):
    """
    Population GCN 2 tầng theo bài báo Parisot et al. (2018)
    """
    def __init__(self, in_features, hidden_dim1=64, hidden_dim2=32, num_classes=2, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = NativeGCNConv(in_features, hidden_dim1)
        self.conv2 = NativeGCNConv(hidden_dim1, hidden_dim2)
            
        self.bn1 = nn.BatchNorm1d(hidden_dim1)
        self.bn2 = nn.BatchNorm1d(hidden_dim2)
        self.classifier = nn.Linear(hidden_dim2, num_classes)
        
    def forward(self, x, adj_norm):
        h = self.conv1(x, adj_norm)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        
        h = self.conv2(h, adj_norm)
        h = self.bn2(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        
        return self.classifier(h)


---
## 🔬 5. Huấn Luyện & Thẩm Định với Fold-wise Feature Selection (Chống Data Leakage)

Để đảm bảo tính trung thực khoa học tuyệt đối:
- **Feature Selection (SelectKBest ANOVA F-test chọn top 2000 features)** chỉ được `fit` trên tập huấn luyện (`train_idx`) của từng Fold.
- Sau đó áp dụng phép biến đổi (`transform`) lên toàn bộ nút để đưa vào GNN.
- Hàm mất mát chỉ tính trên các nút huấn luyện (`train_mask`), đánh giá trên nút kiểm thử (`val_mask`).


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix

def train_eval_fold(model, x, adj_norm, y, train_mask, val_mask, epochs=150, lr=0.01, device='cpu'):
    model = model.to(device)
    x = x.to(device)
    adj_norm = adj_norm.to(device)
    y = y.to(device)
        
    y_train = y[train_mask]
    n_pos = (y_train == 1).sum().item()
    n_neg = (y_train == 0).sum().item()
    weight = torch.tensor([n_pos / max(n_neg, 1), 1.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=15)
    
    best_val_auc = 0.0
    best_metrics = {}
    
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(x, adj_norm)
        loss = criterion(out[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            out_eval = model(x, adj_norm)
            probs = F.softmax(out_eval[val_mask], dim=1)[:, 1].cpu().numpy()
            preds = out_eval[val_mask].argmax(dim=1).cpu().numpy()
            y_val = y[val_mask].cpu().numpy()
            
            acc = accuracy_score(y_val, preds)
            bal_acc = balanced_accuracy_score(y_val, preds)
            try:
                auc = roc_auc_score(y_val, probs)
            except ValueError:
                auc = 0.5
            f1 = f1_score(y_val, preds, zero_division=0)
            
        scheduler.step(auc)
        if auc > best_val_auc:
            best_val_auc = auc
            tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
            best_metrics = {
                'acc': acc, 'bal_acc': bal_acc, 'auc': auc, 'f1': f1,
                'sens': tp / max(tp + fn, 1), 'spec': tn / max(tn + fp, 1)
            }
            
    return best_metrics

def evaluate_population_gcn(features_raw, adj_norm, labels, n_splits=10, k_best=2000, epochs=120, lr=0.01, device='cpu'):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    metrics_history = {'acc': [], 'bal_acc': [], 'auc': [], 'f1': [], 'sens': [], 'spec': []}
    adj_norm_tensor = torch.FloatTensor(adj_norm)
    y_tensor = torch.LongTensor(labels)
    
    print(f"\n{'='*55}\n  ĐÁNH GIÁ {n_splits}-FOLD STRATIFIED CV\n{'='*55}")
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(features_raw, labels)):
        # Fold-wise Feature Selection (Chống Leakage)
        selector = SelectKBest(f_classif, k=min(k_best, features_raw.shape[1]))
        scaler = StandardScaler()
        
        selector.fit(features_raw[train_idx], labels[train_idx])
        X_train_sel = selector.transform(features_raw[train_idx])
        scaler.fit(X_train_sel)
        
        X_all_sel = selector.transform(features_raw)
        X_all_norm = scaler.transform(X_all_sel)
        x_tensor = torch.FloatTensor(X_all_norm)
        
        train_mask = torch.zeros(len(labels), dtype=torch.bool)
        val_mask = torch.zeros(len(labels), dtype=torch.bool)
        train_mask[train_idx] = True
        val_mask[val_idx] = True
        
        model = PopulationGCN(in_features=k_best, hidden_dim1=64, hidden_dim2=32)
        res = train_eval_fold(model, x_tensor, adj_norm_tensor, y_tensor, train_mask, val_mask,
                              epochs=epochs, lr=lr, device=device)
                              
        for m in metrics_history:
            metrics_history[m].append(res[m])
            
        print(f"  Fold {fold+1:2d}/{n_splits} | Acc: {res['acc']*100:.1f}% | "
              f"BalAcc: {res['bal_acc']*100:.1f}% | AUC: {res['auc']*100:.1f}% | F1: {res['f1']*100:.1f}%")
              
    print(f"\nKẾT QUẢ TRUNG BÌNH {n_splits}-FOLD:")
    for m in metrics_history:
        print(f"   {m.upper():<10}: {np.mean(metrics_history[m])*100:.2f}% ± {np.std(metrics_history[m])*100:.2f}%")
        
    return metrics_history


---
## 🚀 6. Thực Thi Thử Nghiệm

Chạy song song:
1. **10-Fold CV**: Đối chiếu với kết quả **70.4%** của bài báo Parisot et al. (2018).
2. **5-Fold CV**: Đối chiếu với kết quả **64.5%** của SVM Baseline repo hiện tại.


In [ ]:
# Chạy mô hình trên GPU nếu có
print(f"Bắt đầu huấn luyện trên thiết bị: {DEVICE}")

# 1. Chạy 10-Fold CV (Chuẩn Parisot et al.)
results_10fold = evaluate_population_gcn(
    features_raw=features,
    adj_norm=adj_norm,
    labels=y,
    n_splits=10,
    k_best=2000,
    epochs=120,
    lr=0.01,
    device=DEVICE
)

# 2. Chạy 5-Fold CV (Chuẩn SVM Baseline)
results_5fold = evaluate_population_gcn(
    features_raw=features,
    adj_norm=adj_norm,
    labels=y,
    n_splits=5,
    k_best=2000,
    epochs=120,
    lr=0.01,
    device=DEVICE
)


---
## 📊 7. Tổng Kết & So Sánh Hiệu Năng Toàn Diện


In [ ]:
summary_df = pd.DataFrame({
    'Mô hình': [
        'Brain Graph GCN (Exp 3a cũ)',
        'Brain Graph GAT (Exp 3b cũ)',
        'Linear SVM Baseline (Repo)',
        'Parisot et al. 2018 (Paper)',
        'Population GCN (Exp 4 - 5 Fold)',
        'Population GCN (Exp 4 - 10 Fold)'
    ],
    'Accuracy (%)': [49.4, 50.6, 64.5, 70.4, 
                     np.mean(results_5fold['acc'])*100, 
                     np.mean(results_10fold['acc'])*100],
    'AUC (%)': [48.6, 52.5, 70.0, 75.0, 
                np.mean(results_5fold['auc'])*100, 
                np.mean(results_10fold['auc'])*100],
    'F1 (%)': [2.6, 5.0, 60.7, 70.0, 
               np.mean(results_5fold['f1'])*100, 
               np.mean(results_10fold['f1'])*100]
})

print(summary_df.to_string(index=False))

# Vẽ biểu đồ so sánh trực quan
plt.figure(figsize=(10, 5))
x_pos = np.arange(len(summary_df))
width = 0.25

plt.bar(x_pos - width, summary_df['Accuracy (%)'], width, label='Accuracy', color='#3498db')
plt.bar(x_pos, summary_df['AUC (%)'], width, label='AUC', color='#2ecc71')
plt.bar(x_pos + width, summary_df['F1 (%)'], width, label='F1 Score', color='#e74c3c')

plt.xticks(x_pos, summary_df['Mô hình'], rotation=25, ha='right')
plt.ylabel('Phần trăm (%)')
plt.title('So Sánh Hiệu Năng: Brain GNN vs. SVM vs. Population GCN (Parisot 2018)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('exp4_benchmark_comparison.png', dpi=120)
plt.show()
